In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy
from src.exception import CustomException
from src.logger import logging
import sys
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
nltk.download("stopwords")
stopword=stopwords.words("english")
from sklearn.feature_extraction.text import CountVectorizer
import re
from dataclasses import dataclass
import os
ps=PorterStemmer()
cv=CountVectorizer()
from sklearn.model_selection import train_test_split
import pickle

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
os.getcwd()

'd:\\Machine_Learning_Projects\\Spam_detection\\research'

In [3]:
os.chdir("../")

In [4]:
os.getcwd()

'd:\\Machine_Learning_Projects\\Spam_detection'

In [5]:
@dataclass
class DataLoaderConfig:
    data_path : str=os.path.join("artifacts","data.csv")

class DataLoading:
    
    def __init__(self):
        self.data_loader_config=DataLoaderConfig()

    def data_loader(self):
        try:
            logging.info("Data Loading in process")

            data=pd.read_csv(r"D:\Machine_Learning_Projects\Spam_detection\data\data_clean.csv")

            os.makedirs(
                os.path.dirname(self.data_loader_config.data_path),
                exist_ok=True
                )
            data.to_csv(
                self.data_loader_config.data_path,index=False
            )
            logging.info("Data Loading Successfully completed")
            return data
        except Exception as e:
            raise e



In [6]:
loader=DataLoading()
data=loader.data_loader()
data.head()

,cleaned,label
0,Weekly Report budget review - Statement our I ...,0
1,Project Update team sync - President series to...,0
2,🔥WIN BIG NOW!! win free urgent offer limited l...,1
3,🔥WIN BIG NOW!! guarantee click now cash offer ...,1
4,Meeting Reminder team sync - Significant prope...,0


In [ ]:
@dataclass
class PreprocessConfig:
        preprocessed_data: str = os.path.join("artifacts", "preprocessing.csv")
        train_path : str = os.path.join("artifacts","train_data.csv")
        test_path : str = os.path.join("artifacts","test_data.csv")

class Preprocessing:
    def __init__(self):
        self.preprocessing_config=PreprocessConfig()
    def clean(self,data):
        try:
            logging.info("Data preprocessing step in process")
            preprocess=re.sub("[^A-Za-z0-9]"," ",str(data))
            preprocess=preprocess.lower().split()
            preprocess=[ps.stem(i) for i in preprocess if i not in stopword]
            return " ".join(preprocess)
        except Exception as e:
            raise e
    
    def cleaned(self,data):
        try:
            data["cleaned"]=data["cleaned"].apply(self.clean)
            os.makedirs(
                os.path.dirname(self.preprocessing_config.preprocessed_data),exist_ok=True
            )
            data.to_csv(
                self.preprocessing_config.preprocessed_data,index=False
            )
            logging.info("Data preprocessing step completed successfully")

            return data
        except Exception as e:
            raise e

    def splitting_dataset(self,data):
        try:
            logging.info("Training and Testing step starts")
            data=pd.read_csv(self.preprocessing_config.preprocessed_data)

            train_data,test_data=train_test_split(data,
                                        random_state=42,
                                        test_size=0.25)

            os.makedirs(
                os.path.dirname(self.preprocessing_config.train_path),exist_ok=True
            )
            train_data.to_csv(
                self.preprocessing_config.train_path,index=False
            )
            os.makedirs(
                os.path.dirname(self.preprocessing_config.test_path),exist_ok=True
            )
            test_data.to_csv(
                self.preprocessing_config.test_path,index=False
            )
            logging.info("Splitting data into training and testing completed successfully")
            return train_data,test_data
        except Exception as e:
            raise e


loader=DataLoading()
data=loader.data_loader()
process = Preprocessing()
processed = process.cleaned(data)
datasets=process.splitting_dataset(processed)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [14]:
@dataclass
class ModelTrainingConfig:
    train_path : str = os.path.join("artifacts","train_data.csv")
    test_path : str = os.path.join("artifacts","test_data.csv")
    model_path : str = os.path.join("artifacts","model.pkl")

class ModelTraining :

    def __init__(self):
        self.model_trainer_config=ModelTrainingConfig()


    def get_data_initialized(self):
        try:
            logging.info("Training and testing Initialization step")
            train_data=pd.read_csv(self.model_trainer_config.train_path)
            test_data=pd.read_csv(self.model_trainer_config.test_path)
            X_train=train_data["cleaned"]
            y_train=train_data["label"]
            X_test=test_data["cleaned"]
            y_test=test_data["label"]
            x_train=cv.fit_transform(X_train)
            x_test=cv.transform(X_test)
            logging.info("Initializing of training and testing data completed successfully")
            return x_train,y_train,x_test,y_test
        except Exception as e:
            raise e

    def model_evaluation(self):
        try:
            logging.info("Model Evaluation Step starts")
            x_train,y_train,x_test,y_test=self.get_data_initialized()
            log=LogisticRegression()
            log.fit(x_train,y_train)
            ypre=log.predict(x_test)
            accuracy=accuracy_score(y_test,ypre)*100
            logging.info(f"Accuracy Score of {log} is : {accuracy}")
            os.makedirs(
                os.path.dirname(self.model_trainer_config.model_path),exist_ok=True
            )
            with open(self.model_trainer_config.model_path,"wb") as file:
                pickle.dump(log,file)
            logging.info("Model Evaluation Step Completed successfully")
        except Exception as e:
            logging.info("Error occur during model execution")
            raise e
model=ModelTraining()
model_training=model.model_evaluation()

In [33]:
@dataclass
class PredictingConfig :
    model_path : str = os.path.join("artifacts","model.pkl")
    preprocess_path : str = os.path.join("artifacts","preprocessing.csv")

class Predicting :
    
    def __init__(self):

        self.prediction=PredictingConfig()
        self.process=Preprocessing()

    def Pred(self,data):

        try :
            logging.info("Prediction Stage is in Process ")

            with open(self.prediction.model_path,"rb") as f:
                model=pickle.load(f)
            logging.info("Model loaded successfully")

            # Data Cleaning Step

            new_cleaning=process.clean(data)

            # Data Converting into numeric

            new_data=cv.transform([new_cleaning])

            # Now Predicting the data

            pre=model.predict(new_data)

            logging.info("Prediction Stage is completed successfully")
            
            return pre

        except Exception as e:
            raise e

In [34]:
user="""Subject: Congratulations! You Won a $1,000 Gift Card!

Dear Customer,

Congratulations! You have been randomly selected as the winner of a $1,000 gift card.

To claim your prize, click the link below and complete your information before the offer expires today.

Claim your reward now!

This is a limited-time opportunity. Don't miss your chance to receive your FREE gift card.

Congratulations once again!
Prize Rewards Team"""
prd=Predicting()
predi=prd.Pred([user])
print(predi)

[1]


In [36]:
user1="""Subject: Interview confirmation

Dear Vivek,

This is to confirm your interview scheduled for Monday at 11:00 AM.

Please join the meeting using the link provided in the previous email. Make sure you have a stable internet connection.

Best regards,
HR Team"""
pre1=prd.Pred([user1])

In [37]:
print(pre1)

[0]
